# Week 3 — Probability refresher: simulation, LLN and CLT

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Simulate common distributions and estimate moments from samples.
- Visualize the Law of Large Numbers (LLN) with simulation.
- Visualize the Central Limit Theorem (CLT) with simulation.
- Connect sampling uncertainty to 'estimating a strategy's mean return'.

## Estimated study time

About 7–9 hours.

## Prerequisites

- Random variables, expectation and variance
- Basic numpy

## External resources

- [MIT OpenCourseWare 18.05 Introduction to Probability and Statistics](https://ocw.mit.edu/courses/18-05-introduction-to-probability-and-statistics-spring-2022/)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### Law of Large Numbers (LLN)

As the sample size $n$ grows, the sample mean $\bar X_n$ converges to the true expected value $\mu$:

$$ \bar X_n = \frac1n\sum_{i=1}^n X_i \xrightarrow[n\to\infty]{} \mu. $$

### Central Limit Theorem (CLT)

Whatever the shape of the population distribution, the **sampling distribution** of the sample mean approaches a normal distribution:

$$ \frac{\bar X_n - \mu}{\sigma/\sqrt n} \xrightarrow{d} N(0, 1). $$

The LLN tells us **where** the mean converges to; the CLT tells us **how much uncertainty** the mean carries at finite $n$ (the standard error $\sigma/\sqrt n$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.math.probability import (
    empirical_moments, running_mean,
    sampling_distribution_of_mean,
    simulate_bernoulli, simulate_normal,
)

rng = np.random.default_rng(2024)

### Simulating distributions and estimating moments

Below we first simulate a normal distribution (continuous), then a Bernoulli distribution (discrete: success/failure). In strategy research a Bernoulli naturally models binary events like 'did we call the direction correctly?'

In [ ]:
# Bernoulli(p=0.55): e.g. an indicator for 'the market goes up tomorrow'; p is the conditional probability
wins = simulate_bernoulli(p=0.55, size=10_000, seed=0)
print('Bernoulli sample mean (≈ p):', round(float(wins.mean()), 3))
print('Bernoulli theoretical variance p(1-p) =', round(0.55 * 0.45, 4))
print('Bernoulli sample variance             =', round(float(wins.var(ddof=1)), 4))

In [ ]:
normal_draws = simulate_normal(mean=0.001, std=0.02, size=10_000, seed=1)
moments = empirical_moments(normal_draws)
print('Estimated moments:', {k: round(v, 6) for k, v in moments.items()})
print('True mean = 0.001, true std = 0.02')

### Visualizing the LLN: convergence of the sample mean

In [ ]:
samples = simulate_normal(mean=0.05, std=1.0, size=20_000, seed=3)
path = running_mean(samples)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(1, len(path) + 1), path, label='Running sample mean')
ax.axhline(0.05, linestyle='--', label='True expected value = 0.05')
ax.set_title('Law of Large Numbers: the sample mean converges as n grows')
ax.set_xlabel('Sample size n')
ax.set_ylabel('Running mean')
ax.legend()
plt.show()

The curve swings wildly at first, then gradually settles onto the true expected value as $n$ increases. **The early swings are exactly sampling uncertainty** — and they are why the average return from a short backtest cannot be taken at face value.

### Visualizing the CLT: the sampling distribution of the mean

In [ ]:
small = sampling_distribution_of_mean(
    population_sampler='exponential', sample_size=5,
    n_experiments=5000, seed=4)
large = sampling_distribution_of_mean(
    population_sampler='exponential', sample_size=200,
    n_experiments=5000, seed=4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(small, bins=40)
axes[0].set_title('Distribution of the sample mean, n=5')
axes[0].set_xlabel('Sample mean')
axes[0].set_ylabel('Count')
axes[1].hist(large, bins=40)
axes[1].set_title('Distribution of the sample mean, n=200')
axes[1].set_xlabel('Sample mean')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()
print('n=5 standard deviation:', round(float(np.std(small)), 4))
print('n=200 standard deviation:', round(float(np.std(large)), 4))

The population is **exponential** (heavily right-skewed), yet the distribution of the sample mean looks more and more normal as $n$ grows, and becomes more and more concentrated (the standard error shrinks). That is the CLT.

### Connecting to strategy returns

Think of 'a strategy's daily return' as a random variable. What we really want to know is its **true expected return $\mu$**, but we can only estimate it with the sample mean from a finite sample. The CLT tells us the uncertainty of that sample mean is $\sigma/\sqrt n$ — the more volatile the returns and the less data we have, the less reliable the estimate.

In [ ]:
# A strategy whose true expected return is 0 — it just got lucky
strategy = simulate_normal(mean=0.0, std=0.01, size=252, seed=99)
mean_est = strategy.mean()
se = strategy.std(ddof=1) / np.sqrt(len(strategy))
print(f'Sample mean daily return over one year = {mean_est:.6f}')
print(f'Standard error = {se:.6f}')
print('The sample mean looks nonzero, but this may well be pure sampling noise.')

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. In your own words, explain what question the LLN answers and what question the CLT answers.
2. Given the standard error $\sigma/\sqrt n$, how many times more samples do you need to cut the standard error in half?
3. Why can the sample mean be approximately normal even when the population is not normal?

### Applied exercises

In [ ]:
# Applied exercise 1: simulate 50000 flips of a fair coin, compute the running mean
# of the proportion of heads, and confirm it converges to 0.5.
flips = rng.integers(0, 2, size=50_000).astype(float)
coin_path = None  # TODO: running_mean(flips)
if coin_path is not None:
    print('Final running proportion:', round(float(coin_path[-1]), 4))

In [ ]:
# Applied exercise 2: run 4000 experiments for each of sample_size = 2, 10, 50, 250,
# print the standard deviation of the sample mean, and watch it shrink with n.
for n in [2, 10, 50, 250]:
    means = None  # TODO: sampling_distribution_of_mean(sample_size=n, n_experiments=4000, seed=0)
    if means is not None:
        print(f'n={n:>3}: std of mean = {np.std(means):.4f}')

### Reflection question

1. Suppose someone hands you a strategy whose 'average daily return over the past year was positive'. Based on this week's material, what reasons would you give for doubting that this proves it truly has a positive expected return?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. What does the Law of Large Numbers (LLN) describe?**
- A. The sample mean converges to the population expected value
- B. The sample mean follows a normal distribution
- C. The variance vanishes
- D. All distributions eventually become normal

**Q2. What does the Central Limit Theorem (CLT) describe?**
- A. The sample mean converges to the expected value
- B. The standardized sample mean approaches a normal distribution
- C. The sample must be very large before the mean can be computed
- D. The population must be normal

**Q3. To cut the standard error σ/√n in half, the sample size must become how many times larger?**
- A. 2 times
- B. 4 times
- C. √2 times
- D. 8 times

**Q4. What is the variance of a Bernoulli(p) random variable?**
- A. p
- B. p²
- C. p(1−p)
- D. 1−p

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: 'bc880fb3b3865aa9', 2: '4711061c6f831038', 3: '856c89cc016c0b43', 4: 'fddeec67bad8338d'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w3-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Conflating the LLN (where the mean converges) with the CLT (how much uncertainty it carries).**
- **Forgetting to set a random seed, making simulation results irreproducible.**
- **Estimating a mean from a tiny sample and treating it as the precise true value.**
- **Concluding the expected value is positive just because the sample mean is positive.**

## After this week, you should be able to

- [ ] Distinguish and explain the LLN and the CLT using simulation.
- [ ] Compute and interpret the standard error of the sample mean.
- [ ] Explain why the average of short-horizon strategy returns cannot be taken at face value.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.